In [ ]:
# Stage 1
# Check setup / config / API key is correct, basic invocation
# .env, config.py
from src.config import llm

print(llm.invoke('Say hello').content)

In [ ]:
# Stage 2 - Test DB setup w/ basic query, first interaction builds singleton
# Data in data.py, hardcoded for now, can be extended to hit any API / DB
# Vector store singleton created at first import (python lazy loaded)
# vector store enables semantic search, uses openAI embeddings model in config.py
# search product catalog is a tool in tools.py, natural language query returns products

# Embedding - numerical representation of text, transformer model, we using OpenAI embeddings
#             each vector has same # dimensions, things closer together have similar meaning
# data.py, rag.py, tools.py
from src.tools import search_snackstack_menu

print(search_snackstack_menu.invoke({"query": "I am in the mood for italian food, got any?"}))

In [ ]:
# Stage 3 - Product Agent Subgraph
# can add debug as example
from langchain_core.messages import HumanMessage, SystemMessage
from src.nodes import menu_subgraph, MENU_PROMPT

result = menu_subgraph.invoke({'messages': [
    SystemMessage(content=MENU_PROMPT),
    HumanMessage(content='I am in the mood for italian food, got any? I am also vegan FYI')]} #,debug=True
)

# LLM translates the human query to a slightly simpler plain text query to the tool as an arg based on context, check logs
print(result['messages'][-1].content)


In [ ]:
# Stage 4 - Support agent subgraph

from langchain_core.messages import HumanMessage, SystemMessage
from src.nodes import order_subgraph, ORDER_PROMPT

result = order_subgraph.invoke({'messages': [
    SystemMessage(content=ORDER_PROMPT),
    HumanMessage(content='Please give me order tracking status for 201?')]})

print(result['messages'][-1].content)

In [ ]:
# Stage 5, orchestration / multi-agent

# This is simplified version of what is in the project orchestrator, used to demo
# defined in nodes.orchestrator_node

from src.config import llm
from src.state import ClassificationResult

# models can be requested to align their response with a schema, great for production workflows
# makes things more predictable / maintainable
# ensures output can be easily parsed for further processing
# task configuration / execution
structured_llm = llm.with_structured_output(ClassificationResult)
structured_response = structured_llm.invoke('Classify: My order ORD201 is late show me alternative food options')

print('Mixed:',
      [task.agent for task in structured_response.tasks],
      'synthesis:', structured_response.requires_synthesis)


In [ ]:
print(structured_response)

In [ ]:
# Stage 6
# main.py graph execution
# takes prior subgraphs, execute within node called 'agent' that does simple translation
# executes the multi-agent flow (drawn above)

from langchain_core.messages import HumanMessage
from src.graph import snackstack_graph

result = snackstack_graph.invoke(
 {'messages': [HumanMessage(content='My order ORD201 is late show me alternative food options')],
 'user_query': 'My order ORD201 is late show me alternative food options'},
 {'configurable': {'thread_id': 'test-006'}}) # this is for memory, thread_id used as sessionId

print(result['final_answer'])

In [ ]:
# Stage7
# support agent HITL, gather more info

from langchain_core.messages import HumanMessage
from langgraph.types import Command
from src.graph import snackstack_graph

cfg = {'configurable': {'thread_id': 'customerId'}} # sessionId
result = snackstack_graph.invoke(
    {'messages': [HumanMessage(content='Where is my order?')],
    'user_query': 'Where is my order?'}, cfg , debug=True
)

if '__interrupt__' in result and result['__interrupt__']:
    agent_ask = result['__interrupt__'][0].value
    print('Agent asks:', agent_ask)
    hitl_input = input(agent_ask)
    hitl_result = snackstack_graph.invoke(Command(resume=hitl_input), cfg)
    #hitl_result = snackstack_graph.invoke(Command(resume='ORD201'), cfg)

    print(hitl_result['final_answer'])

# run again with same sessionId, can see the 'values' context keeps prior state

In [ ]:
history = list(snackstack_graph.get_state_history(cfg))
for snapshot in history: print(snapshot, "\n")
# see snapshots / next state when resume empty = ended
# snapshot created after each node execution
# can see interrupts=(Interrupt(value= in snapshot history (bottom up)

In [ ]:
# Stage 8
# main.py / REPL (read -> eval -> print loop)
# graph.py is main graph
import sys
from src.main import main

# show diff of parallel vs. sequential e.g. search for:
# my order 201 is late, show me similar food options
# off the bat

sys.argv = ['src.main']
main()

In [ ]:
# Stage 9
# voice.py
# switch to webcam mic
import sys
from src.main import main

sys.argv = ['src.main', '--voice']
main()